In [13]:
import os
import json
import time
from openai import OpenAI

# ─── Configuration ─────────────────────────────────────────────────────────────
API_KEY = "sk-REPLACE_WITH_YOUR_KEY"
MODEL = "gpt-4o"
CONTEXT_TOKEN_LIMIT = 4000

# ─── File Structure ────────────────────────────────────────────────────────────
# chapters/         folder containing chapter_1.txt, chapter_2.txt, ...
# outline.txt       detailed outline of the book
# book_meta.json    metadata about book (title, audience, chapters, words_per_chapter)
# helper_memory.json   conversational memory

# Initialize OpenAI client
client = OpenAI(api_key=API_KEY)

def save_json(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

def load_json(path, default):
    if os.path.exists(path): return json.load(open(path, 'r', encoding='utf-8'))
    return default

# Ensure directories
os.makedirs('chapters', exist_ok=True)

# Load or initialize meta and memory
meta = load_json('book_meta.json', {})
memory = load_json('helper_memory.json', [])

# On first run, collect book parameters
if not meta:
    print("Welcome! Let's set up your book project.")
    meta['title'] = input("Book title: ").strip()
    meta['audience'] = input("Audience: ").strip()
    meta['chapters'] = int(input("Number of chapters: ").strip())
    meta['words_per_chapter'] = int(input("Approx words per chapter: ").strip())
    save_json('book_meta.json', meta)
    print("Generating initial outline, please wait...")
    prompt = (
        f"Create a detailed outline for a {meta['chapters']}-chapter book titled '{meta['title']}' "
        f"for {meta['audience']}. Each chapter summary should be ~{meta['words_per_chapter']//10} words and detailed enough to write the chapter content."
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":"You are an expert book outlining assistant."},
            {"role":"user","content":prompt}
        ],
        max_tokens=CONTEXT_TOKEN_LIMIT,
        temperature=0.7
    )
    outline = resp.choices[0].message.content
    with open('outline.txt','w',encoding='utf-8') as f: f.write(outline)
    memory.append({'role':'assistant','content':outline})
    save_json('helper_memory.json', memory)
    print("Outline generated and saved to outline.txt.")

print("\nType your instruction for the helper bot. Prefix with 'exit' to quit.")

# Main interactive loop
while True:
    cmd = input("➤ ").strip()
    if cmd.lower() == 'exit':
        print("Goodbye!")
        break
    # build context: include meta, outline, and chapters if referenced
    context = []
    # always include metadata
    context.append(f"BOOK META: {json.dumps(meta)}")
    # include outline if outline referenced or for any structural commands
    if any(w in cmd.lower() for w in ['outline','chapter','condense','expand','write','edit','feedback']):
        outline = open('outline.txt','r',encoding='utf-8').read()
        context.append(f"OUTLINE:\n{outline}")
    # if chapters referenced explicitly, load all chapters
    if any(w in cmd.lower() for w in ['chapter','feedback','clarity','tone','consistency']):
        chapters_text = ''
        for i in range(1, meta['chapters']+1):
            path = f'chapters/chapter_{i}.txt'
            if os.path.exists(path):
                chapters_text += f"\n\n### Chapter {i}\n" + open(path,'r',encoding='utf-8').read()
        if chapters_text:
            context.append(f"CHAPTERS:\n{chapters_text}")
    # send to helper
    messages = [{'role':'system','content':'You are a central book helper with full access when prompted.'}]
    # feed last few memory turns
    for m in memory[-6:]: messages.append(m)
    # feed context as user
    for c in context: messages.append({'role':'user','content':c})
    # final user prompt
    messages.append({'role':'user','content':cmd})
    # call API
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=CONTEXT_TOKEN_LIMIT,
        temperature=0.7
    )
    reply = resp.choices[0].message.content
    print(reply)
    # store memory
    memory.append({'role':'user','content':cmd})
    memory.append({'role':'assistant','content':reply})
    save_json('helper_memory.json', memory)
    # if reply contains chapter generation, save new chapters
    # (the helper should output markers like "### Chapter X")
    if '### Chapter' in reply:
        parts = reply.split('### Chapter')[1:]
        for part in parts:
            header, body = part.split('\n',1)
            num = header.strip().split()[0]
            path = f'chapters/chapter_{num}.txt'
            with open(path,'w',encoding='utf-8') as f:
                f.write(body.strip())
        print("Updated chapters saved.")



👋 Welcome to Interactive Book Writer!
Commands:
  /topic <title> |/audience <who> |/chapters <n>   → setup book
  /outline         → (re)generate outline
  /chap <num>      → draft a single chapter
  /ok              → generate *all* unwritten chapters
  /edit <num>      → edit that chapter
  /show <num>      → print chapter from memory
  /exit            → quit & save session

Just type anything else to talk to the assistant.



➤   hey new chatbot checking that this works


Hello! Yes, it works. How can I assist you today?


➤   ok well lets say i would like to write a book


That's fantastic! Writing a book is an exciting journey. Let's start with the basics. Do you have a specific genre or idea in mind for your book? If not, we can brainstorm some concepts together. Additionally, it would be helpful to know if you have any characters or a particular setting you're considering.


...

I apologize for any confusion. I don't have direct access to external documents or files, but I can certainly help you develop and refine your ideas here. If you have specific parts of your book or chapter that you'd like to discuss or get feedback on, feel free to share them with me. You can describe the content, themes, characters, or any particular sections you're working on, and I'll do my best to assist you with advice and suggestions!


➤   oh well i guess i messed up my code :(


No worries! It happens to everyone. If you're comfortable sharing, feel free to describe the issue you're facing with your code. I can help troubleshoot or provide suggestions to resolve the problem. Whether it's a specific error message, unexpected behavior, or something else, I'm here to help!


➤   /ok


KeyboardInterrupt: Interrupted by user